# Preprocessing

This notebook uses the original CSV as the main dataset because it contains the raw fields needed for cleaning, feature selection, and feature-group experiments. The `_ML.csv` file is useful as a reference, but it is already transformed and has a different row count and target distribution.


## Load Data


In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/youtube_shorts_tiktok_trends_2025.csv")
df_ml = pd.read_csv("../data/youtube_shorts_tiktok_trends_2025.csv_ML.csv")

print("Original shape:", df.shape)
print("ML shape:", df_ml.shape)

df.head()


Original shape: (48079, 58)
ML shape: (50000, 32)


,platform,country,region,language,category,hashtag,title_keywords,author_handle,sound_type,music_track,...,traffic_source,is_weekend,row_id,engagement_total,like_rate,dislike_rate,engagement_per_1k,engagement_like_rate,engagement_comment_rate,engagement_share_rate
0,TikTok,Jp,Asia,ja,Gaming,#Lifestyle,Night Routine — College,NextVision,trending,8bit loop,...,External,1,2e681528d17a1fe1986857942536ec27,30317,0.086159,0.004004,120.069,0.086159,0.012555,0.007830
1,TikTok,Se,Europe,sv,Food,#Sports,Morning Routine — College,DailyVlogsDiego,trending,Street vibe,...,Search,0,2e35fa0b2978b9cae635839c1d4e9e74,30577,0.085298,0.002421,113.005,0.085298,0.007850,0.007791
2,TikTok,Za,Africa,en,Art,#Workout,Night Routine — College,BeyondHub,licensed,Gallery pad,...,External,1,0d88a011235a82244995ef52961f9502,503,0.049154,0.001625,68.111,0.049154,0.004469,0.005146
3,TikTok,Kr,Asia,ko,News,#Esports,Best Settings for Fortnite,NextHub,original,Neutral piano,...,Search,1,e15cff7621ed3f9eb9d2c97c841be0f3,7828,0.086257,0.003164,108.156,0.086257,0.011205,0.005292
4,TikTok,Au,Oceania,en,Beauty,#Comedy,When your friend is Beginners,LucasOfficial,licensed,Soft glam loop,...,ForYou,1,d696b4f0a50ea70e7cb5021be7e198ec,1171,0.051441,0.001175,72.400,0.051441,0.004204,0.004142


## Compare Original vs ML Version

The row count and label distribution are different, so we should not treat these as interchangeable versions of the same table.


In [2]:
print("Original target distribution")
display(df["trend_label"].value_counts(normalize=True).sort_index())

print("ML target distribution")
display(df_ml["trend_label"].value_counts(normalize=True).sort_index())


Original target distribution


trend_label
declining    0.249652
rising       0.251711
seasonal     0.252626
stable       0.246012
Name: proportion, dtype: float64

ML target distribution


trend_label
declining    0.10196
rising       0.25000
seasonal     0.09412
stable       0.55392
Name: proportion, dtype: float64

## Basic Data Checks


In [3]:
summary = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "n_unique": df.nunique(),
    }
)

print("Duplicate rows:", df.duplicated().sum())
display(summary.sort_values("missing", ascending=False))


Duplicate rows: 0


,dtype,missing,n_unique
platform,object,0,2
country,object,0,30
region,object,0,6
language,object,0,19
category,object,0,19
hashtag,object,0,41
title_keywords,object,0,137
author_handle,object,0,720
sound_type,object,0,3
music_track,object,0,61


## Clean Values

This keeps cleaning simple: trim whitespace, lowercase categorical text, parse the approximate publish date, and create simple date parts.


In [4]:
clean = df.copy()

text_columns = clean.select_dtypes(include=["object", "string"]).columns
for col in text_columns:
    clean[col] = clean[col].astype("string").str.strip().str.lower()

clean["publish_date_approx"] = pd.to_datetime(clean["publish_date_approx"], errors="coerce")
clean["publish_month"] = clean["publish_date_approx"].dt.month
clean["publish_day"] = clean["publish_date_approx"].dt.day

clean.head()


,platform,country,region,language,category,hashtag,title_keywords,author_handle,sound_type,music_track,...,row_id,engagement_total,like_rate,dislike_rate,engagement_per_1k,engagement_like_rate,engagement_comment_rate,engagement_share_rate,publish_month,publish_day
0,tiktok,jp,asia,ja,gaming,#lifestyle,night routine — college,nextvision,trending,8bit loop,...,2e681528d17a1fe1986857942536ec27,30317,0.086159,0.004004,120.069,0.086159,0.012555,0.007830,1,4
1,tiktok,se,europe,sv,food,#sports,morning routine — college,dailyvlogsdiego,trending,street vibe,...,2e35fa0b2978b9cae635839c1d4e9e74,30577,0.085298,0.002421,113.005,0.085298,0.007850,0.007791,1,1
2,tiktok,za,africa,en,art,#workout,night routine — college,beyondhub,licensed,gallery pad,...,0d88a011235a82244995ef52961f9502,503,0.049154,0.001625,68.111,0.049154,0.004469,0.005146,1,5
3,tiktok,kr,asia,ko,news,#esports,best settings for fortnite,nexthub,original,neutral piano,...,e15cff7621ed3f9eb9d2c97c841be0f3,7828,0.086257,0.003164,108.156,0.086257,0.011205,0.005292,1,3
4,tiktok,au,oceania,en,beauty,#comedy,when your friend is beginners,lucasofficial,licensed,soft glam loop,...,d696b4f0a50ea70e7cb5021be7e198ec,1171,0.051441,0.001175,72.400,0.051441,0.004204,0.004142,1,4


## Choose Columns

These drops are intentionally conservative. Text columns can be useful, but they need a separate NLP approach. Trend-duration fields are excluded from the first ML table because they may leak the target.


In [5]:
TARGET = "trend_label"

text_id_or_notes_columns = [
    "row_id",
    "source_hint",
    "notes",
    "title",
    "title_keywords",
    "tags",
    "sample_comments",
    "author_handle",
    "music_track",
    "publish_date_approx",
    "year_month",
]

possible_leakage_columns = [
    "trend_duration_days",
    "trend_type",
    "engagement_velocity",
]

drop_columns = [TARGET] + text_id_or_notes_columns + possible_leakage_columns
drop_columns = [col for col in drop_columns if col in clean.columns]

X = clean.drop(columns=drop_columns)
y = clean[TARGET]

print("Dropped columns:")
display(drop_columns)

print("X shape:", X.shape)
print("y shape:", y.shape)
display(X.head())


Dropped columns:


['trend_label',
 'row_id',
 'source_hint',
 'notes',
 'title',
 'title_keywords',
 'tags',
 'sample_comments',
 'author_handle',
 'music_track',
 'publish_date_approx',
 'year_month',
 'trend_duration_days',
 'trend_type',
 'engagement_velocity']

X shape: (48079, 45)
y shape: (48079,)


,platform,country,region,language,category,hashtag,sound_type,week_of_year,duration_sec,views,...,is_weekend,engagement_total,like_rate,dislike_rate,engagement_per_1k,engagement_like_rate,engagement_comment_rate,engagement_share_rate,publish_month,publish_day
0,tiktok,jp,asia,ja,gaming,#lifestyle,trending,1,40,252497,...,1,30317,0.086159,0.004004,120.069,0.086159,0.012555,0.007830,1,4
1,tiktok,se,europe,sv,food,#sports,trending,1,18,270580,...,0,30577,0.085298,0.002421,113.005,0.085298,0.007850,0.007791,1,1
2,tiktok,za,africa,en,art,#workout,licensed,1,22,7385,...,1,503,0.049154,0.001625,68.111,0.049154,0.004469,0.005146,1,5
3,tiktok,kr,asia,ko,news,#esports,original,1,36,72377,...,1,7828,0.086257,0.003164,108.156,0.086257,0.011205,0.005292,1,3
4,tiktok,au,oceania,en,beauty,#comedy,licensed,1,35,16174,...,1,1171,0.051441,0.001175,72.400,0.051441,0.004204,0.004142,1,4


## Optional Synthetic-Column Removal Check

The notes flag several fields as synthetic, synthetic-derived, or risky for leakage. This optional section builds a no-synthetic feature set for sensitivity checks without changing the main processed files. If this analysis is not needed, this section can be commented out or deleted.


In [ ]:
synthetic_or_caution_features = [
    # Already removed in the main pipeline, listed here for auditability.
    "row_id",
    "source_hint",
    "notes",
    "title",
    "title_keywords",
    "tags",
    "sample_comments",
    "author_handle",
    "music_track",
    "trend_duration_days",
    "trend_type",
    "engagement_velocity",
    # Still present in the main processed set, but synthetic or synthetic-derived.
    "dislikes",
    "like_dislike_ratio",
    "dislike_rate",
    "avg_watch_time_sec",
    "completion_rate",
]

present_synthetic_or_caution = [col for col in synthetic_or_caution_features if col in X.columns]
X_no_synthetic = X.drop(columns=present_synthetic_or_caution)
y_no_synthetic = y.copy()

print("Synthetic/caution features removed from current X:")
display(present_synthetic_or_caution)
print("Original X shape:", X.shape)
print("No-synthetic X shape:", X_no_synthetic.shape)
display(X_no_synthetic.head())


Synthetic/caution features removed from current X:


['dislikes',
 'like_dislike_ratio',
 'dislike_rate',
 'title_length',
 'has_emoji',
 'avg_watch_time_sec',
 'completion_rate',
 'creator_avg_views']

Original X shape: (48079, 45)
No-synthetic X shape: (48079, 37)


,platform,country,region,language,category,hashtag,sound_type,week_of_year,duration_sec,views,...,traffic_source,is_weekend,engagement_total,like_rate,engagement_per_1k,engagement_like_rate,engagement_comment_rate,engagement_share_rate,publish_month,publish_day
0,tiktok,jp,asia,ja,gaming,#lifestyle,trending,1,40,252497,...,external,1,30317,0.086159,120.069,0.086159,0.012555,0.007830,1,4
1,tiktok,se,europe,sv,food,#sports,trending,1,18,270580,...,search,0,30577,0.085298,113.005,0.085298,0.007850,0.007791,1,1
2,tiktok,za,africa,en,art,#workout,licensed,1,22,7385,...,external,1,503,0.049154,68.111,0.049154,0.004469,0.005146,1,5
3,tiktok,kr,asia,ko,news,#esports,original,1,36,72377,...,search,1,7828,0.086257,108.156,0.086257,0.011205,0.005292,1,3
4,tiktok,au,oceania,en,beauty,#comedy,licensed,1,35,16174,...,foryou,1,1171,0.051441,72.400,0.051441,0.004204,0.004142,1,4


## Feature Groups

Use these groups later for RQ experiments. The group definitions are written once here and saved with the processed files.


In [7]:
feature_groups = {
    "metadata": [
        "platform",
        "country",
        "region",
        "language",
        "category",
        "hashtag",
        "sound_type",
        "device_type",
        "device_brand",
        "traffic_source",
    ],
    "temporal": [
        "week_of_year",
        "upload_hour",
        "publish_dayofweek",
        "publish_period",
        "event_season",
        "season",
        "is_weekend",
        "publish_month",
        "publish_day",
    ],
    "content_basic": [
        "genre",
        "duration_sec",
        "title_length",
        "has_emoji",
    ],
    "creator": [
        "creator_avg_views",
        "creator_tier",
    ],
    "engagement_observed": [
        "views",
        "likes",
        "comments",
        "shares",
        "saves",
        "dislikes",
        "engagement_total",
        "engagement_rate",
        "comment_ratio",
        "share_rate",
        "save_rate",
        "like_dislike_ratio",
        "like_rate",
        "dislike_rate",
        "engagement_per_1k",
        "engagement_like_rate",
        "engagement_comment_rate",
        "engagement_share_rate",
        "avg_watch_time_sec",
        "completion_rate",
    ],
}

feature_groups = {
    group: [col for col in cols if col in X.columns]
    for group, cols in feature_groups.items()
}

display(pd.DataFrame([
    {"feature_group": group, "n_raw_columns": len(cols), "raw_columns": ", ".join(cols)}
    for group, cols in feature_groups.items()
]))


,feature_group,n_raw_columns,raw_columns
0,metadata,10,"platform, country, region, language, category,..."
1,temporal,9,"week_of_year, upload_hour, publish_dayofweek, ..."
2,content_basic,4,"genre, duration_sec, title_length, has_emoji"
3,creator,2,"creator_avg_views, creator_tier"
4,engagement_observed,20,"views, likes, comments, shares, saves, dislike..."


## Train / Validation / Test Split

The split is stratified so each set keeps roughly the same class balance.


In [8]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp,
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

display(y_train.value_counts(normalize=True).sort_index())
display(y_val.value_counts(normalize=True).sort_index())
display(y_test.value_counts(normalize=True).sort_index())


Train: (33655, 45) (33655,)
Validation: (7212, 45) (7212,)
Test: (7212, 45) (7212,)


trend_label
declining    0.249651
rising       0.251701
seasonal     0.252622
stable       0.246026
Name: proportion, dtype: Float64

trend_label
declining    0.249584
rising       0.251803
seasonal     0.252634
stable       0.245979
Name: proportion, dtype: Float64

trend_label
declining    0.249723
rising       0.251664
seasonal     0.252634
stable       0.245979
Name: proportion, dtype: Float64

## Encode Features

Categorical columns are converted to dummy variables here, so RQ1, RQ2, and RQ3 can read numeric CSVs directly without building preprocessing pipelines again.


In [9]:
categorical_columns = X_train.select_dtypes(include=["object", "string", "category"]).columns.tolist()
numeric_columns = [col for col in X_train.columns if col not in categorical_columns]

# Boolean columns are model-ready once converted to 0/1 values.
bool_columns = X_train[numeric_columns].select_dtypes(include=["bool"]).columns.tolist()
for data_part in [X_train, X_val, X_test]:
    for col in bool_columns:
        data_part[col] = data_part[col].astype("int8")

numeric_medians = X_train[numeric_columns].median(numeric_only=True)

def encode_split(X_part):
    numeric_part = X_part[numeric_columns].fillna(numeric_medians)
    categorical_part = pd.get_dummies(
        X_part[categorical_columns],
        dtype="int8",
    )
    return pd.concat([numeric_part, categorical_part], axis=1)

X_train_encoded = encode_split(X_train)
X_val_encoded = encode_split(X_val).reindex(columns=X_train_encoded.columns, fill_value=0)
X_test_encoded = encode_split(X_test).reindex(columns=X_train_encoded.columns, fill_value=0)

X_train = X_train_encoded
X_val = X_val_encoded
X_test = X_test_encoded

print("Categorical columns encoded:", len(categorical_columns))
print("Numeric columns kept:", len(numeric_columns))
print("Encoded train shape:", X_train.shape)
print("Encoded validation shape:", X_val.shape)
print("Encoded test shape:", X_test.shape)

display(X_train.head())


Categorical columns encoded: 16
Numeric columns kept: 29
Encoded train shape: (33655, 203)
Encoded validation shape: (7212, 203)
Encoded test shape: (7212, 203)


,week_of_year,duration_sec,views,likes,comments,shares,saves,engagement_rate,upload_hour,dislikes,...,device_brand_pixel,device_brand_samsung,device_brand_vivo,device_brand_xiaomi,traffic_source_external,traffic_source_following,traffic_source_foryou,traffic_source_home,traffic_source_search,traffic_source_suggested
18885,15,21,134960,8055,970,1003,1290,0.083862,10,230,...,0,0,0,0,0,0,1,0,0,0
19451,15,15,98468,4239,666,368,636,0.060009,7,96,...,0,1,0,0,0,0,1,0,0,0
37955,28,58,294112,17051,2376,1506,1101,0.074917,15,201,...,0,1,0,0,0,0,1,0,0,0
36480,27,63,160850,4305,608,481,283,0.035294,21,315,...,0,0,0,0,0,0,0,1,0,0
32543,24,38,211401,21203,3061,2540,1410,0.133462,17,910,...,0,0,0,0,0,0,1,0,0,0


## Save


In [10]:
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

raw_to_group = {
    column: group
    for group, columns in feature_groups.items()
    for column in columns
}

feature_metadata_rows = []
for encoded_feature in X_train.columns:
    original_feature = encoded_feature
    for categorical_column in categorical_columns:
        if encoded_feature.startswith(f"{categorical_column}_"):
            original_feature = categorical_column
            break

    feature_metadata_rows.append({
        "feature": encoded_feature,
        "original_feature": original_feature,
        "feature_group": raw_to_group.get(original_feature, "other"),
    })

feature_metadata = pd.DataFrame(feature_metadata_rows)
feature_group_table = feature_metadata[["feature_group", "feature"]].copy()

train = X_train.copy()
train[TARGET] = y_train

validation = X_val.copy()
validation[TARGET] = y_val

test = X_test.copy()
test[TARGET] = y_test

train.to_csv(processed_dir / "train.csv", index=False)
validation.to_csv(processed_dir / "validation.csv", index=False)
test.to_csv(processed_dir / "test.csv", index=False)
feature_metadata.to_csv(processed_dir / "feature_metadata.csv", index=False)
feature_group_table.to_csv(processed_dir / "feature_groups.csv", index=False)

print("Saved files to", processed_dir)
print("Saved feature metadata:", feature_metadata.shape)
display(feature_metadata.head())


Saved files to ..\data\processed
Saved feature metadata: (203, 3)


,feature,original_feature,feature_group
0,week_of_year,week_of_year,temporal
1,duration_sec,duration_sec,content_basic
2,views,views,engagement_observed
3,likes,likes,engagement_observed
4,comments,comments,engagement_observed
